# Flat LMDP examples

This notebook runs the current four-room examples in small, inspectable steps.

In [ ]:
from pathlib import Path
import runpy
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

# This works whether Jupyter starts in the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from andrew_mlmdp import (
    Maze,
    controlled_dynamics,
    desirability_grid,
    plot_controlled_dynamics,
    plot_trajectory,
    sample_rollout,
    solve_desirability,
)

## Load the four-room maze

In [ ]:
maze = Maze.from_file(PROJECT_ROOT / "mazes" / "four_rooms.txt")
goal = (10, 9)

print(f"maze shape: {maze.shape}")
print(f"free states: {len(maze.free_cells)}")
print(f"goal state: {maze.state_index(goal)}")

## Exact desirability

In [ ]:
desirability = solve_desirability(maze, goal)
grid = desirability_grid(maze, desirability)

fig, ax = plt.subplots(figsize=(7, 6))
# Log scaling reveals structure among the very small values far from the goal.
positive_values = grid[np.isfinite(grid) & (grid > 0.0)]
log_scale = LogNorm(vmin=positive_values.min(), vmax=positive_values.max())
image = ax.imshow(grid, cmap="viridis", norm=log_scale)
ax.plot(goal[1], goal[0], marker="*", color="red", markersize=13)
ax.set_title("Exact desirability")
ax.set_xlabel("column")
ax.set_ylabel("row")
fig.colorbar(image, ax=ax, label="desirability (log scale)")
plt.show()

## Controlled next-state probabilities

In [ ]:
controlled = controlled_dynamics(maze, desirability)
ax = plot_controlled_dynamics(maze, controlled, goal=goal)
plt.show()

## Sample a rollout

In [ ]:
start = (0, 0)
trajectory = sample_rollout(
    maze,
    controlled,
    start,
    goal,
    seed=7,
)

print(f"reached goal: {trajectory[-1] == goal}")
print(f"steps: {len(trajectory) - 1}")
ax = plot_trajectory(maze, trajectory, goal=goal)
plt.show()

## Run the figure scripts

These scripts repeat the controlled-dynamics and rollout examples and save their PNGs in `output/`.

In [ ]:
example_scripts = [
    "plot_flat_policy.py",
    "plot_sample_rollout.py",
]

for script_name in example_scripts:
    runpy.run_path(
        str(PROJECT_ROOT / "experiments" / script_name),
        run_name="__main__",
    )